# Symbolic Heisenberg equations of motion (adjoint Lindblad)

Given an operator $A$, a Hamiltonian $\hat H$, and a list of collapse operators $\mathbf L = [c_1, c_2, \dots]$, this notebook computes (Breuer & Petruccione, *The Theory of Open Quantum Systems*, Eq. 3.84, $\hbar = 1$)

$$\frac{d\langle A\rangle}{dt} = -i\big\langle [A,H]\big\rangle + \sum_k\Big\langle c_k^{\dagger}Ac_k - \tfrac12\{c_k^{\dagger}c_k, A\}\Big\rangle$$

(same as derivations in `three_level_system_gorshkov.tex`)

**Algebra used**
* Atomic states $g, s, e$ are orthonormal: $|a\rangle\langle b|\,|c\rangle\langle d| = \delta_{bc}\,|a\rangle\langle d|$.
* Cavity mode: $[\hat a, \hat a^\dagger] = 1$; every result is put in normal order $(\hat a^\dagger)^m \hat a^n$.
* Atomic and cavity operators commute with each other.
* Coefficients (couplings, $\alpha_{\rm in}(t)$, $\beta(t)$, ...) are ordinary sympy expressions; `dag()` complex-conjugates them. Declare symbols `real`/`positive` where appropriate so that $g^* = g$.

**Notation (same as the notes):** 
- $\hat\sigma_{\mu\nu} = |\nu\rangle\langle\mu|$
- `sig('g','e')` $=\hat\sigma_{ge} = |e\rangle\langle g|$ 
- Use `ketbra(k, b)` $= |k\rangle\langle b|$ if you prefer to write projectors directly. 
- Output is always printed in $\hat\sigma_{\mu\nu}$ notation.

Collapse operators are taken with the rate already absorbed (e.g. $c_1 = \kappa_{ge}\hat a$), as in the notes. Optionally pass `rates=[...]` to multiply each dissipator by $\gamma_k$.

## Operator algebra engine

In [ ]:
import sympy as sp
from IPython.display import display, Math

from math import comb, factorial
from collections import defaultdict

LEVELS = ('g', 's', 'e')

class Op:
    """Sum of terms  coeff * (atomic |k><b| or 1) * (a^dag)^m a^n   (normal ordered).
    terms: {(atom, (m, n)): sympy coeff}, atom = None (identity) or (ket, bra)."""
    def __init__(self, terms=None):
        self.terms = {}
        for k, c in (terms or {}).items():
            c = sp.expand(c)
            if c != 0:
                self.terms[k] = c

    @staticmethod
    def _lift(x):
        return x if isinstance(x, Op) else Op({(None, (0, 0)): sp.sympify(x)})

    def __add__(self, other):
        other = Op._lift(other)
        t = defaultdict(lambda: sp.Integer(0), self.terms)
        for k, c in other.terms.items():
            t[k] += c
        return Op(t)
    __radd__ = __add__
    def __neg__(self):
        return Op({k: -c for k, c in self.terms.items()})
    def __sub__(self, other):
        return self + (-Op._lift(other))
    def __rsub__(self, other):
        return Op._lift(other) - self

    def __mul__(self, other):
        other = Op._lift(other)
        t = defaultdict(lambda: sp.Integer(0))
        for (at1, (m, n)), c1 in self.terms.items():
            for (at2, (p, q)), c2 in other.terms.items():
                # atomic part: |a><b| |c><d| = delta_bc |a><d|
                if at1 is None:
                    at = at2
                elif at2 is None:
                    at = at1
                elif at1[1] == at2[0]:
                    at = (at1[0], at2[1])
                else:
                    continue
                # bosonic part: a^n (a^dag)^p = sum_k C(n,k) C(p,k) k! (a^dag)^(p-k) a^(n-k)
                for k in range(min(n, p) + 1):
                    w = comb(n, k) * comb(p, k) * factorial(k)
                    t[(at, (m + p - k, n + q - k))] += w * c1 * c2
        return Op(t)
    def __rmul__(self, other):
        return Op._lift(other) * self

    def dag(self):
        return Op({((at[1], at[0]) if at else None, (n, m)): sp.conjugate(c)
                   for (at, (m, n)), c in self.terms.items()})

    def __repr__(self):
        return repr(to_sympy(self))
    def _repr_latex_(self):
        return '$' + sp.latex(to_sympy(self)) + '$'


def dag(X):
    return Op._lift(X).dag()

def ketbra(k, b):
    assert k in LEVELS and b in LEVELS
    return Op({((k, b), (0, 0)): 1})

def sig(mu, nu):
    """Notebook convention: sigma_{a b} = |b><a|  (e.g. sigma_ge = |e><g|, sigma_eg = |g><e|)."""
    return ketbra(nu, mu)

a = Op({(None, (0, 1)): 1})
ad = a.dag()
I = Op._lift(1)

def comm(X, Y):
    return X * Y - Y * X

def acomm(X, Y):
    return X * Y + Y * X


# ---------- display ----------
def _op_symbol(at, mn):
    m, n = mn
    parts = []
    if at is not None:
        ket, bra = at
        parts.append(r'\hat\sigma_{%s%s}' % (bra, ket))   # |ket><bra| = sigma_{bra ket}
    if m:
        parts.append(r'\hat a^{\dagger}' + ('' if m == 1 else '^{%d}' % m))
    if n:
        parts.append(r'\hat a' + ('' if n == 1 else '^{%d}' % n))
    if not parts:
        return sp.Integer(1)
    return sp.Symbol(r'\,'.join(parts), commutative=False)

def to_sympy(X, simplify=False):
    X = Op._lift(X)
    out = sp.Integer(0)
    for (at, mn), c in X.terms.items():
        cc = sp.factor(c) if simplify else c
        out += cc * _op_symbol(at, mn)
    return out


# ---------- Heisenberg / adjoint Lindblad equation ----------
def heisenberg(A, H, L, rates=None):
    """dA/dt = -i[A,H] + sum_k gamma_k ( c_k^dag A c_k - 1/2 {c_k^dag c_k, A} )   (hbar = 1)."""
    A, H = Op._lift(A), Op._lift(H)
    rates = [1] * len(L) if rates is None else rates
    pieces = {'-i[A,H]': -sp.I * comm(A, H)}
    total = pieces['-i[A,H]']
    for k, (c, r) in enumerate(zip(L, rates), 1):
        c = Op._lift(c)
        jump = r * (c.dag() * A * c)
        anti = -sp.Rational(1, 2) * r * acomm(c.dag() * c, A)
        pieces[f'c_{k}^dag A c_{k}'] = jump
        pieces[f'-1/2{{c_{k}^dag c_{k}, A}}'] = anti
        total = total + jump + anti
    pieces['total'] = total
    return total, pieces

In [ ]:
# ---------- weak-excitation (single-excitation) truncation ----------
_ORDER = {'g': 0, 's': 1, 'e': 1}

def _term_order(at, mn):
    o = sum(mn)                                # each photon operator is O(eps)
    if at is not None:
        o += _ORDER[at[0]] + _ORDER[at[1]]     # <|k><b|> = c_b^* c_k
    return o

def weak_excitation(X, order, ground_to_one=True):
    """Keep only terms of order <= `order` in eps for a state c_g|g> + c_s|s> + c_e|e> (+ photons)
    with c_g ~ 1 and c_s, c_e, <a> = O(eps).  If ground_to_one, replace sigma_gg -> 1."""
    X = Op._lift(X)
    t = {}
    for (at, mn), c in X.terms.items():
        if _term_order(at, mn) > order:
            continue
        if ground_to_one and at == ('g', 'g'):
            at = None
        t[(at, mn)] = t.get((at, mn), 0) + c
    return Op(t)


# ---------- pretty printing ----------
def show_eom(A, H, L, rates=None, pieces=True, weak=False):
    """ Display dA/dt.  
    - pieces=True shows each term of the adjoint Lindblad equation.
    - weak=True additionally shows the weak-excitation-limit result."""
    A = Op._lift(A)
    total, parts = heisenberg(A, H, L, rates)
    Alat = sp.latex(to_sympy(A))
    if pieces:
        display(Math(r'-i[%s, \hat H] = %s' % (Alat, sp.latex(to_sympy(parts['-i[A,H]'])))))
        for k, c in enumerate(L, 1):
            clat = sp.latex(to_sympy(c))
            display(Math(r'c_{%d} = %s:\quad c_{%d}^\dagger %s\, c_{%d} = %s,\qquad '
                         r'-\tfrac12\{c_{%d}^\dagger c_{%d}, %s\} = %s'
                         % (k, clat, k, Alat, k, sp.latex(to_sympy(parts[f'c_{k}^dag A c_{k}'])),
                            k, k, Alat, sp.latex(to_sympy(parts[f'-1/2{{c_{k}^dag c_{k}, A}}'])))))
    display(Math(r'\boxed{\frac{d}{dt}%s = %s}' % (Alat, sp.latex(to_sympy(total)))))
    if weak:
        order = max(_term_order(at, mn) for (at, mn) in A.terms)
        w = weak_excitation(total, order)
        display(Math(r'\text{weak excitation: }\ \frac{d\langle %s\rangle}{dt} \approx %s'
                     % (Alat, sp.latex(to_sympy(w)))))
    return total

## Example 1: one cavity on the $ge$ arm

$$\hat H = \big(\kappa_{ge}\alpha_{\rm in}(t)\hat a^{\dagger}_{ge} + \text{h.c.}\big)
+ \big(g_{se}\beta(t)\hat\sigma_{se} + \text{h.c.}\big)
+ g_{ge}\big(\hat a_{ge}\hat\sigma_{ge} + \text{h.c.}\big)
+ \Delta\hat\sigma_{ee},\qquad
\mathbf{L} = \big[\kappa_{ge}\hat a_{ge},\; -g_{ge,nc}\,\hat\sigma_{eg}\big]$$

In [3]:
t = sp.symbols('t', real=True)
kap, gge, gse, Delta, gnc = sp.symbols(r'\kappa_{ge} g_{ge} g_{se} \Delta g_{ge\,nc}', positive=True)
alpha = sp.Function(r'\alpha_{in}')(t)   # complex, time dependent
beta  = sp.Function(r'\beta')(t)

H_cav = (kap*alpha*ad + dag(kap*alpha*ad)) \
      + (gse*beta*sig('s','e') + dag(gse*beta*sig('s','e'))) \
      + gge*(a*sig('g','e') + dag(a*sig('g','e'))) \
      + Delta*sig('e','e')
L_cav = [kap*a, -gnc*sig('e','g')]

display(Math(r'\hat H = ' + sp.latex(to_sympy(H_cav))))

<IPython.core.display.Math object>

### $A = \hat a_{ge}$ (Gorshkov Eq. 3)

In [4]:
_ = show_eom(a, H_cav, L_cav)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### $A = \hat\sigma_{eg} = |g\rangle\langle e|$ (Gorshkov's $P$, Eq. 4)

In [5]:
_ = show_eom(sig('e','g'), H_cav, L_cav, weak=True)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### $A = \hat\sigma_{sg} = |g\rangle\langle s|$ (Gorshkov's $S$, Eq. 5)

In [6]:
_ = show_eom(sig('s','g'), H_cav, L_cav, weak=True)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Populations, e.g. $A = \hat\sigma_{ss}$, $\hat\sigma_{ee}$

In [ ]:
_ = show_eom(sig('s','s'), H_cav, L_cav, pieces=False)
_ = show_eom(sig('e','e'), H_cav, L_cav, pieces=False)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Example 2: free space

$$\hat H = \Delta\hat\sigma_{ee}
+ g_{ge}\big(\alpha_{\rm in}(t)\,\hat\sigma_{ge} + h.c.\big)
+ \big(g_{se}\beta(t)\,\hat\sigma_{se} + h.c.\big),\qquad
\mathbf{L} = \Big[-g^{*}_{ge}\hat\sigma_{eg},\; -\tfrac{1}{\sqrt C}\,g^{*}_{ge}\hat\sigma_{eg}\Big],\quad g_{ge} = \sqrt{2\gamma C}$$

Here $\alpha_{\rm in}$ is a c-number drive (no cavity mode). The weak-excitation result should reproduce $-[\gamma(1+C)+i\Delta]\hat\sigma_{eg}$.

In [8]:
gam, C = sp.symbols(r'\gamma C', positive=True)
g_fs = sp.sqrt(2*gam*C)

H_fs = Delta*sig('e','e') \
     + g_fs*(alpha*sig('g','e') + dag(alpha*sig('g','e'))) \
     + (gse*beta*sig('s','e') + dag(gse*beta*sig('s','e')))
L_fs = [-g_fs*sig('e','g'), -g_fs/sp.sqrt(C)*sig('e','g')]

P_fs = show_eom(sig('e','g'), H_fs, L_fs, weak=True)
S_fs = show_eom(sig('s','g'), H_fs, L_fs, weak=True)

# sigma_eg = |g><e| has key (('g','e'), (0,0))
display(Math(r'\text{coefficient of }\hat\sigma_{eg}: ' + sp.latex(sp.factor(P_fs.terms[(('g','e'), (0,0))]))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Numerical check of the algebra

Builds explicit matrices ($3\times3$ atom $\otimes$ truncated Fock space), plugs random numbers into all symbols, and compares the symbolic result against $-i[A,H] + \sum_k (c_k^\dagger A c_k - \tfrac12\{c_k^\dagger c_k, A\})$ computed directly with matrices. Comparison is made on the low-photon block, away from the Fock truncation edge.

In [9]:
import numpy as np

def to_matrix(X, subs, N=10):
    X = Op._lift(X)
    am = np.diag(np.sqrt(np.arange(1, N)), 1)
    idx = {l: i for i, l in enumerate(LEVELS)}
    M = np.zeros((3*N, 3*N), complex)
    for (at, (m, n)), c in X.terms.items():
        A_at = np.eye(3) if at is None else np.outer(np.eye(3)[idx[at[0]]], np.eye(3)[idx[at[1]]])
        B = np.linalg.matrix_power(am.conj().T, m) @ np.linalg.matrix_power(am, n)
        M += complex(sp.N(c.subs(subs))) * np.kron(A_at, B)
    return M

def check(A, H, L, N=10, nmax=4, seed=0):
    rng = np.random.default_rng(seed)
    syms = set()
    for X in [A, H, *L]:
        for c in Op._lift(X).terms.values():
            syms |= c.atoms(sp.Symbol) | c.atoms(sp.Function)
    subs = {}
    for s in syms:
        if isinstance(s, sp.Symbol) and s.is_real:
            subs[s] = rng.uniform(0.5, 2) if s.is_positive else rng.uniform(-2, 2)
        elif s.func is not sp.conjugate:
            subs[s] = complex(rng.normal(), rng.normal())
    total, _ = heisenberg(A, H, L)
    Am, Hm, Lm = to_matrix(A, subs, N), to_matrix(H, subs, N), [to_matrix(c, subs, N) for c in L]
    direct = -1j*(Am@Hm - Hm@Am) + sum(c.conj().T@Am@c - 0.5*(c.conj().T@c@Am + Am@c.conj().T@c) for c in Lm)
    keep = [i*N + n for i in range(3) for n in range(nmax + 1)]
    return np.abs((to_matrix(total, subs, N) - direct)[np.ix_(keep, keep)]).max()

for name, (H, L) in {'cavity': (H_cav, L_cav), 'free space': (H_fs, L_fs)}.items():
    for A in [a, sig('e','g'), sig('s','g'), sig('s','s'), sig('e','e'), ad*a]:
        print(f'{name:10s}  A = {str(to_sympy(A)):28s} max error = {check(A, H, L):.2e}')

cavity      A = \hat a                       max error = 1.78e-15
cavity      A = \hat\sigma_{eg}              max error = 0.00e+00
cavity      A = \hat\sigma_{sg}              max error = 0.00e+00
cavity      A = \hat\sigma_{ss}              max error = 0.00e+00
cavity      A = \hat\sigma_{ee}              max error = 0.00e+00
cavity      A = \hat a^{\dagger}\,\hat a     max error = 1.07e-14
free space  A = \hat a                       max error = 4.44e-16
free space  A = \hat\sigma_{eg}              max error = 4.44e-16
free space  A = \hat\sigma_{sg}              max error = 0.00e+00
free space  A = \hat\sigma_{ss}              max error = 0.00e+00
free space  A = \hat\sigma_{ee}              max error = 8.88e-16
free space  A = \hat a^{\dagger}\,\hat a     max error = 8.88e-16


## Your own system

Build $H$, $\mathbf L$ (a list), and $A$ from `sig(mu, nu)`, `ketbra(k, b)`, `a`, `ad`, `dag(...)`, `+`, `*`, and sympy coefficients, then call `show_eom(A, H, L)`. `heisenberg(A, H, L)` returns `(total, pieces)` as `Op` objects if you want to manipulate the result: `to_sympy(op)` converts to a sympy expression, `weak_excitation(op, order)` truncates, and `op.terms` is the raw `{(atomic |k><b| or None, (m, n)): coefficient}` dictionary.

Note: the engine has a single bosonic mode `a`; a second cavity (e.g. on the $se$ arm) would need a second mode added to the term keys.

In [14]:
A = sig('e','g')
H = H_cav
L = L_cav
_ = show_eom(A, H, L, pieces=True)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [15]:
A = sig('e','g')
H = H_cav
L = [kap*a, -gnc*sig('e','g'), -gse*sig('e','s')]

_ = show_eom(A, H, L)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [16]:
A = sig('s','s')
H = H_cav
L = [kap*a, -gnc*sig('e','g'), -gse*sig('e','s')]

_ = show_eom(A, H, L)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>